# SinoNom OCR — Phase 1: Asynchronous Image Scraper

**SinoNom Document Intelligence Project**  
**An Nam Nhất Thống Chí (HVH_004)**

---

This notebook handles **Phase 1** of the pipeline: downloading raw scanned manuscript page images for _An Nam Nhất Thống Chí_ (volume 664) from the Nom Foundation Digital Library and saving them locally.


## 0 · Environment Setup


In [7]:
# ─── Detect environment ────────────────────────────────────────────────────
import os
import sys

IN_COLAB = "google.colab" in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")
print(f"Python {sys.version}")

Running in Google Colab: False
Python 3.12.7 | packaged by conda-forge | (main, Oct  4 2024, 15:57:01) [Clang 17.0.6 ]


In [8]:
# ─── Google Drive Mount & Project Path Discovery (Colab only) ────────────────
if IN_COLAB:
    import os
    import subprocess
    from pathlib import Path

    repo_name = "SinoNomViet_Transliteration_OCR"
    repo_url = f"https://github.com/khang3004/{repo_name}.git"
    candidate_files = [
        "src/sinonom_ocr/data_scraper.py",
        "src/sinonom_ocr/spatial_layout_engine.py",
        "src/sinonom_ocr/alignment_validator.py",
    ]

    def has_project_files(path: Path) -> bool:
        return all((path / f).exists() for f in candidate_files)

    found_root = None

    # 1. Check if files are already in current directory or a subfolder of /content
    cwd = Path(os.getcwd())
    if has_project_files(cwd):
        found_root = cwd
    else:
        for p in Path("/content").glob("**/src/sinonom_ocr/data_scraper.py"):
            if has_project_files(p.parent.parent.parent):
                found_root = p.parent.parent.parent
                break

    # 2. If not found locally, try to find them on Google Drive
    if not found_root:
        drive_mounted = os.path.exists("/content/drive/MyDrive")
        if not drive_mounted:
            print("❓ Google Drive is not mounted. Mounting now...")
            try:
                from google.colab import drive

                drive.mount("/content/drive")
                drive_mounted = True
            except Exception as e:
                print(f"⚠️ Google Drive mount failed or skipped: {e}")

        if drive_mounted:
            drive_paths = [
                Path("/content/drive/MyDrive/SinoNomOCR/HVH_004"),
                Path(f"/content/drive/MyDrive/{repo_name}"),
                Path("/content/drive/MyDrive/SinoNomVietnamese_OCR_Project"),
                Path("/content/drive/MyDrive/SinoNomViet_Transliteration_OCR"),
            ]
            for dp in drive_paths:
                if dp.exists() and has_project_files(dp):
                    found_root = dp
                    print(f"🎯 Found project files in Drive: {found_root}")
                    break

            if not found_root:
                print("🔍 Searching Google Drive for project files...")
                for p in Path("/content/drive/MyDrive").glob("**/src/sinonom_ocr/data_scraper.py"):
                    if has_project_files(p.parent.parent.parent):
                        found_root = p.parent.parent.parent
                        print(f"🎯 Found project files in Drive subfolder: {found_root}")
                        break

    # 3. If still not found, clone the repository from GitHub
    if not found_root:
        print("📁 Project files not found. Cloning repository from GitHub...")
        try:
            subprocess.run(["git", "clone", repo_url, f"/content/{repo_name}"], check=True)
            found_root = Path(f"/content/{repo_name}")
            print("✅ Repository cloned successfully.")
        except Exception as e:
            print(f"❌ Failed to clone repository: {e}")

    if found_root:
        PROJECT_ROOT = str(found_root)
        os.chdir(PROJECT_ROOT)
        print(f"🎯 Project root set to: {PROJECT_ROOT}")
        print(f"🔄 Changed working directory to: {os.getcwd()}")
    else:
        PROJECT_ROOT = "/content"
        print("❌ ERROR: Could not locate project root.")
else:
    # Local Jupyter running in notebooks/ directory
    PROJECT_ROOT = os.path.abspath("..")
    os.chdir(PROJECT_ROOT)
    print(f"🎯 Local project root: {PROJECT_ROOT}")
    print(f"🔄 Changed working directory to: {os.getcwd()}")

🎯 Local project root: /Users/KhangDS/Programing/HCMUS_Code/NaturalLanguageProcessing_code
🔄 Changed working directory to: /Users/KhangDS/Programing/HCMUS_Code/NaturalLanguageProcessing_code


In [9]:
# ─── Install package & dependencies ────────────────────────────────────────
# Installs the package in editable mode, which resolves all dependencies from pyproject.toml
if IN_COLAB:
    # Install the package in editable mode to register sinonom_ocr package
    %pip install -q -e .
    print("✅ Package and dependencies installed on Colab.")
else:
    print("✅ Local environment is managed by uv (run 'make install' if needed).")

✅ Local environment is managed by uv (run 'make install' if needed).


In [10]:
# ─── Path configuration ────────────────────────────────────────────────────
import sys
from pathlib import Path

ROOT = Path(PROJECT_ROOT)

# Add src/ directory to sys.path so we can import our sinonom_ocr package
SRC_PATH = ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Detect Google Drive paths (Local Mac or Colab)
LOCAL_DRIVE_PATH = Path("/Users/KhangDS/Library/CloudStorage/GoogleDrive-gausseuler159357@gmail.com/My Drive/SinoNomOCR/HVH_004")
COLAB_DRIVE_PATH = Path("/content/drive/MyDrive/SinoNomOCR/HVH_004")

# Determine active directories
if IN_COLAB and COLAB_DRIVE_PATH.exists():
    print("✨ Using Google Drive directories on Google Colab.")
    DATA_DIR = COLAB_DRIVE_PATH / "data"
    OUTPUT_DIR = COLAB_DRIVE_PATH / "output"
elif LOCAL_DRIVE_PATH.exists():
    print("✨ Using Google Drive directories on local macOS.")
    DATA_DIR = LOCAL_DRIVE_PATH / "data"
    OUTPUT_DIR = LOCAL_DRIVE_PATH / "output"
else:
    print("✨ Using local repository directories.")
    DATA_DIR = ROOT / "data"
    OUTPUT_DIR = ROOT / "output"

# Output directories
RAW_IMAGES_DIR = DATA_DIR / "raw_images"
OUTPUT_XML_DIR = OUTPUT_DIR / "xml"
OUTPUT_XLSX_DIR = OUTPUT_DIR / "excel"
DICT_DIR = DATA_DIR / "dicts"

for d in [RAW_IMAGES_DIR, OUTPUT_XML_DIR, OUTPUT_XLSX_DIR, DICT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Copy reference dictionary files if missing from target DICT_DIR
repo_dicts_dir = ROOT / "data" / "dicts"
if repo_dicts_dir.exists() and DICT_DIR != repo_dicts_dir:
    import shutil
    for dic_file in repo_dicts_dir.glob("*.dic"):
        target_file = DICT_DIR / dic_file.name
        if not target_file.exists():
            shutil.copy(dic_file, target_file)
            print(f"📋 Copied reference dictionary {dic_file.name} to {target_file}")

print("Directory structure initialized.")

✨ Using Google Drive directories on local macOS.
Directory structure initialized.


---

## 1 · Import Scraper Module


In [11]:
import logging

# Pipeline modules
from sinonom_ocr.data_scraper import ScraperConfig, run_scraper

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)-8s | %(name)s | %(message)s",
)

print("✅ Scraper module imported successfully.")

✅ Scraper module imported successfully.


---

## 3 · Scrape Raw Images

Download all page images from the Nom Foundation volume page.

> 💡 **Skip this cell** if images are already downloaded or you are using mock data.


In [12]:
# ─── Scraper configuration ─────────────────────────────────────────────────
# Automatically enable scraping if no images exist in the RAW_IMAGES_DIR
has_existing_images = any(RAW_IMAGES_DIR.glob("page_*.jpg"))
SCRAPE_ENABLED = not has_existing_images  # Set to True/False to override manually

if SCRAPE_ENABLED:
    scraper_cfg = ScraperConfig(
        base_url="https://lib.nomfoundation.org/collection/1/volume/664/",
        output_dir=RAW_IMAGES_DIR,
        max_workers=6,
        request_delay=1.0,  # Polite: 1 s between requests
        max_retries=3,
        timeout=45,
    )
    page_map = run_scraper(scraper_cfg)
    print(f"\n✅ Scraped {len(page_map)} pages → {RAW_IMAGES_DIR}")
else:
    # List already-downloaded images
    page_map = {int(p.stem.split("_")[-1]): p for p in sorted(RAW_IMAGES_DIR.glob("page_*.jpg"))}
    print(f"Found {len(page_map)} existing images in {RAW_IMAGES_DIR}")
    if not page_map:
        print("  ⚠️  No images found — proceeding with mock OCR data.")

Found 107 existing images in /Users/KhangDS/Library/CloudStorage/GoogleDrive-gausseuler159357@gmail.com/My Drive/SinoNomOCR/HVH_004/data/raw_images
